# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Lane chosen: Refresh / Content Opportunity Scoring


---


**Why this Lane?**
1. Numbers already show the scale: 93.8% of clients (30 of 32) have at least one declining page, and 54.2% of all 30,000 pages (16,262) are currently declining. No content team can manually inspect over half their inventory every week. There must be a model which automatically detects & ranks the pages requiring refresh/modification.
2. The question is about prioritization- "which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?"
3. Its output is a ranked queue which can be used by the content team to work on, and decide which declining pages to prioritize. Output is deliverable that the real team can use.
4. A model that learns from multiple weighted signals at once (the way the starter random forest already does) captures relationships a single hand-written threshold structurally cannot.

**The decision and action this produces**
1. Who acts: a content strategist/editor with limited weekly capacity.
2. The action: review the top N pages in the queue first — refresh, expand, protect, prune, or monitor, per the reason code attached.
3. The cost of getting it wrong: wasted editor time on a false positive (low-value page reviewed for nothing), or a real, high-traffic decliner sitting unseen and continuing to lose visibility (a false negative).

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

-> Lane: Refresh / Content Opportunity Scoring

I am choosing this lane because it directly matches the shape of the starter dataset: the file is literally named content_refresh_anonymized.csv.Refresh prioritization is also a naturally decision-shaped problem: a content team can't refresh 16,262 pages at once, so ranking them by expected impact is inherently more useful than a raw declining/not-declining flag. Compared to Ranking Signal Analysis (more exploratory, less clearly tied to one action) or CTR Scoring (a narrower slice of the same underlying data), this lane lets me build toward the reference pipeline's own benchmark (Precision@50 ≈ 0.24 → 0.74), giving me a clear, measurable bar for whether my model is actually better than a simple hand-written rule.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, json
!git clone https://github.com/Vedika1304-05/.git
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Claim 1: Is 54% decline real, and not just noise on dead pages? ---
n_total = len(df)
n_declining = (df["trend_direction"] == "down").sum()
pct_declining = n_declining / n_total * 100
visible_declining = df[(df["trend_direction"]=="down") & (df["impressions_90d"]>=100)]
print(f"Declining pages: {n_declining:,} ({pct_declining:.1f}%)")
print(f"Declining AND with real traffic: {len(visible_declining):,} "
      f"({len(visible_declining)/n_total*100:.1f}% of all pages)")

# --- Claim 2: Is the cost of a wrong call asymmetric (uneven)? ---
declining = df[df["trend_direction"]=="down"]
median_impr = declining["impressions_90d"].median()
high = declining[declining["impressions_90d"] >= median_impr]
low  = declining[declining["impressions_90d"] <  median_impr]
ratio = high["impressions_90d"].median() / max(low["impressions_90d"].median(), 1)
print(f"High-traffic decliners vs low-traffic decliners: {ratio:.1f}x more impressions")

# --- Claim 3: Does staleness alone explain decline? ---
med_days_declining = declining["days_since_last_update"].median()
med_days_all = df["days_since_last_update"].median()
print(f"Median days since update — declining: {med_days_declining:.0f} | all pages: {med_days_all:.0f}")

# --- Claim 4: Model vs. hand-written rule (from the trained pipeline) ---
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {base:.3f} | Random forest Precision@50: {rf:.3f} "
      f"-> {rf/base:.2f}x improvement")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

In [ ]:
# This cell is for CODE (numbers, a query, a check).
import os

repo_name = "flyrank-internship-ml"
repo_url = "https://github.com/Vedika1304-05/flyrank-internship-ml.git"

if not os.path.exists(repo_name):
    !git clone {repo_url}

%cd {repo_name}
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# How many clients does this problem affect? (breadth check)
n_clients_total = df["client_id"].nunique()
declining = df[df["trend_direction"] == "down"]
n_clients_declining = declining["client_id"].nunique()

print(f"Total clients in dataset: {n_clients_total}")
print(f"Clients with at least one declining page: {n_clients_declining} "
      f"({n_clients_declining/n_clients_total*100:.1f}%)")

# How does refresh-relevant content_type break down?
print("\nContent type distribution:")
print(df["content_type"].value_counts())
print("\nContent type distribution among DECLINING pages:")
print(declining["content_type"].value_counts())

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 140 (delta 50), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.92 MiB | 13.01 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/flyrank-internship-ml
Total clients in dataset: 32
Clients with at least one declining page: 30 (93.8%)

Content type distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Content type distribution among DECLINING pages:
content_type
keyword article       15262
feedly article          601
comparison article      399
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**decision improved:** which declining pages a content team should refresh first, given limited weekly refresh capacity.

**who acts on it:** a content strategist or SEO manager working through a prioritized queue — not an automated system. This is a decision-support tool, not an autonomous action.

**The unit of analysis:** one page (content_id) per row — the same grain as the raw data.

**cost of wrong recommendations:**
1. false positives: the content team wastes time in updating not so important pages, thereby leading to inefficient usage of time and resources.
2. false negatives: a genuinely high-opportunity page ranked low, so it never gets refreshed. The page keeps declining unnoticed, and the client silently loses search traffic/revenue over the following months.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Traffic value at stake — what's the size of the mistake if you misjudge priority?
# "High-value decliner" = declining AND already has meaningful traffic
high_value_decliners = declining[declining["impressions_90d"] >= declining["impressions_90d"].median()]
low_value_decliners = declining[declining["impressions_90d"] < declining["impressions_90d"].median()]

print(f"Declining pages ABOVE median traffic: {len(high_value_decliners):,}")
print(f"  → median impressions_90d: {high_value_decliners['impressions_90d'].median():,.0f}")
print(f"Declining pages BELOW median traffic: {len(low_value_decliners):,}")
print(f"  → median impressions_90d: {low_value_decliners['impressions_90d'].median():,.0f}")

# Spread in traffic size = spread in cost of getting priority wrong
print(f"\nRatio of high-value to low-value median traffic: "
      f"{high_value_decliners['impressions_90d'].median() / max(low_value_decliners['impressions_90d'].median(),1):.1f}x")

Declining pages ABOVE median traffic: 8,133
  → median impressions_90d: 3,831
Declining pages BELOW median traffic: 8,129
  → median impressions_90d: 179

Ratio of high-value to low-value median traffic: 21.4x


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Traffic value at stake — what's the size of the mistake if you misjudge priority?
# "High-value decliner" = declining AND already has meaningful traffic
high_value_decliners = declining[declining["impressions_90d"] >= declining["impressions_90d"].median()]
low_value_decliners = declining[declining["impressions_90d"] < declining["impressions_90d"].median()]

print(f"Declining pages ABOVE median traffic: {len(high_value_decliners):,}")
print(f"  → median impressions_90d: {high_value_decliners['impressions_90d'].median():,.0f}")
print(f"Declining pages BELOW median traffic: {len(low_value_decliners):,}")
print(f"  → median impressions_90d: {low_value_decliners['impressions_90d'].median():,.0f}")

# Spread in traffic size = spread in cost of getting priority wrong
print(f"\nRatio of high-value to low-value median traffic: "
      f"{high_value_decliners['impressions_90d'].median() / max(low_value_decliners['impressions_90d'].median(),1):.1f}x")

Declining pages ABOVE median traffic: 8,133
  → median impressions_90d: 3,831
Declining pages BELOW median traffic: 8,129
  → median impressions_90d: 179

Ratio of high-value to low-value median traffic: 21.4x


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Scale of the opportunity: how many pages are declining?
declining_pct = (df["trend_direction"] == "down").mean() * 100
print(f"Declining pages: {(df['trend_direction'] == 'down').sum():,} of {len(df):,} "
      f"({declining_pct:.1f}%)")

# 2. Is there enough traffic/volume behind these to matter?
measurable = df[df["impressions_90d"] >= 100]
print(f"Pages with ≥100 impressions/90d: {len(measurable):,} "
      f"({len(measurable)/len(df)*100:.1f}%)")

# 3. How stale is the content that's declining? (refresh angle check)
declining = df[df["trend_direction"] == "down"]
print(f"Median days since last update (declining pages): "
      f"{declining['days_since_last_update'].median():.0f}")
print(f"Median days since last update (all pages): "
      f"{df['days_since_last_update'].median():.0f}")

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 135 (delta 46), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.92 MiB | 18.68 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/flyrank-internship-ml
Declining pages: 16,262 of 30,000 (54.2%)
Pages with ≥100 impressions/90d: 22,006 (73.4%)
Median days since last update (declining pages): 20
Median days since last update (all pages): 20


In [ ]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Scale of the opportunity: how many pages are declining?
declining_pct = (df["trend_direction"] == "down").mean() * 100
print(f"Declining pages: {(df['trend_direction'] == 'down').sum():,} of {len(df):,} "
      f"({declining_pct:.1f}%)")

# 2. Is there enough traffic/volume behind these to matter?
measurable = df[df["impressions_90d"] >= 100]
print(f"Pages with ≥100 impressions/90d: {len(measurable):,} "
      f"({len(measurable)/len(df)*100:.1f}%)")

# 3. How stale is the content that's declining? (refresh angle check)
declining = df[df["trend_direction"] == "down"]
print(f"Median days since last update (declining pages): "
      f"{declining['days_since_last_update'].median():.0f}")
print(f"Median days since last update (all pages): "
      f"{df['days_since_last_update'].median():.0f}")

Declining pages: 16,262 of 30,000 (54.2%)
Pages with ≥100 impressions/90d: 22,006 (73.4%)
Median days since last update (declining pages): 20
Median days since last update (all pages): 20


**Key findings:**
1. 16,262 of 30,000 pages (54.2%) are currently declining, matching the dictionary's stated label distribution exactly — confirming the data loaded correctly and that this is a large, non-niche population worth prioritizing.
2. Median days since last update is identical for declining pages and all pages (20 days each). This is a genuinely useful negative result. It means staleness alone does not obviously separate decliners from non-decliners in this sample, so my refresh-opportunity model can't rely on "how old is this content" as a standalone signal. This pushes me toward combining staleness with other signals (position trend, competition, content type) rather than treating recency as the primary driver

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check: confirm the starter data has no raw client names or URLs -
# only anonymized/hashed IDs. This is what lets me say "decision-support",
# not "here is Client X's exact page" - and backs the self-check below.

id_cols = ["content_id", "client_id"]
for col in id_cols:
    sample = df[col].dropna().astype(str).head(3).tolist()
    print(f"{col} sample values: {sample}")

# None of these should look like a real URL or a real company name -
# they should look like opaque hashed tokens (e.g. 'client_f369cb89fc').
looks_like_url = df["content_id"].astype(str).str.contains("http", case=False).any()
print(f"\nAny content_id containing a raw URL? {looks_like_url}")

content_id sample values: ['content_304f48230142', 'content_a1fb4e703a9e', 'content_9aa793d4d895']
client_id sample values: ['client_f369cb89fc', 'client_4e07408562', 'client_7f2253d7e2']

Any content_id containing a raw URL? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.